In [0]:
# Notebook responsável pela transformação da camada Silver para Gold.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

print("Ambiente Gold preparado com sucesso.")

In [0]:
# Carrega as tabelas tratadas da camada Silver.

df_info = spark.table("workspace.silver.tb_info_filmes")
df_financeiro = spark.table("workspace.silver.tb_financeiro_filmes")
df_metricas = spark.table("workspace.silver.tb_metricas_engajamento")
df_avaliacoes = spark.table("workspace.silver.tb_avaliacoes_usuarios")
df_generos = spark.table("workspace.silver.tb_generos")
df_pessoas_empresas = spark.table("workspace.silver.tb_pessoas_empresas")
df_cotacao = spark.table("workspace.silver.tb_cotacao_dolar")

print("Tabelas Silver carregadas com sucesso.")


In [0]:
#  Cria e grava a dimensão de filmes na Gold

from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Carrega as informações tratadas dos filmes diretamente da Silver
df_info = spark.table("workspace.silver.tb_info_filmes")

# Cria uma chave substituta para cada filme
janela_filmes = Window.orderBy("id")

dim_movies = (
    df_info
    .select(
        "id",
        "tconst",
        "title",
        "original_title",
        "original_language",
        "release_date",
        "runtime",
        "status",
        "overview",
        "tagline"
    )
    .dropDuplicates(["id"])
    .withColumn(
        "sk_movie_id",
        row_number().over(janela_filmes)
    )
    .select(
        "sk_movie_id",
        col("id").alias("movie_id"),
        "tconst",
        "title",
        "original_title",
        "original_language",
        "release_date",
        "runtime",
        "status",
        "overview",
        "tagline"
    )
)

# Grava a dimensão de filmes como tabela Delta
(
    dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_movies")
)

print(
    "Dimensão de filmes gravada com sucesso:",
    spark.table("workspace.gold.dim_movies").count(),
    "registros"
)

display(spark.table("workspace.gold.dim_movies").limit(20))


In [0]:
# Cria a dimensão de gêneros.

janela_generos = Window.orderBy("genero")

dim_genres = (
    df_generos
    .select("genero")
    .filter(
        col("genero").isNotNull() &
        (trim(col("genero")) != "")
    )
    .dropDuplicates(["genero"])
    .withColumn(
        "sk_genre_id",
        row_number().over(janela_generos)
    )
    .select(
        "sk_genre_id",
        col("genero").alias("genre")
    )
)

display(dim_genres.orderBy("sk_genre_id"))




In [0]:
# Cria e grava a dimensão de pessoas

from pyspark.sql import Window
from pyspark.sql.functions import col, trim, length, row_number

# Carrega os dados da Silver

df_pessoas_empresas = spark.table(
    "workspace.silver.tb_pessoas_empresas"
)

# Seleciona os nomes de pessoas

df_people_base = (
    df_pessoas_empresas
    .filter(col("tipo").isin("ator", "diretor", "roteirista"))
    .select(trim(col("nome")).alias("person_name"))
    .filter(
        col("person_name").isNotNull() &
        (col("person_name") != "") &
        (length(col("person_name")) <= 100)
    )
    .dropDuplicates(["person_name"])
)

# Cria a chave de cada pessoa

dim_people = (
    df_people_base
    .withColumn(
        "sk_person_id",
        row_number().over(Window.orderBy("person_name"))
    )
    .select("sk_person_id", "person_name")
)

# Grava a dimensão na Gold

(
    dim_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_people")
)

print(
    "Dimensão de pessoas gravada:",
    spark.table("workspace.gold.dim_people").count(),
    "registros"
)


In [0]:
# Cria e grava a dimensão de empresas

from pyspark.sql import Window
from pyspark.sql.functions import col, trim, row_number

# Carrega as empresas tratadas na Silver

df_empresas_base = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(col("tipo") == "empresa")
    .select(trim(col("nome")).alias("company_name"))
    .filter(
        col("company_name").isNotNull() &
        (col("company_name") != "")
    )
    .dropDuplicates(["company_name"])
)

# Cria uma chave substituta para cada empresa

janela_empresas = Window.orderBy("company_name")

dim_companies = (
    df_empresas_base
    .withColumn(
        "sk_company_id",
        row_number().over(janela_empresas)
    )
    .select("sk_company_id", "company_name")
)

# Grava a dimensão na Gold

(
    dim_companies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_companies")
)

print(
    "Dimensão de empresas gravada com sucesso:",
    spark.table("workspace.gold.dim_companies").count(),
    "registros"
)

display(spark.table("workspace.gold.dim_companies").limit(10))


In [0]:
# Cria a dimensão de gêneros e a ligação com filmes

from pyspark.sql import Window
from pyspark.sql.functions import col, trim, lower, row_number

# Carrega as tabelas
df_filmes = spark.table("workspace.gold.dim_movies")
df_filmes_generos = spark.table("workspace.silver.tb_generos")

# Cria a dimensão de gêneros
df_generos_base = (
    df_filmes_generos
    .select(trim(col("genero")).alias("genre_name"))
    .filter(
        col("genre_name").isNotNull() &
        (col("genre_name") != "")
    )
    .dropDuplicates(["genre_name"])
)

dim_genres = (
    df_generos_base
    .withColumn(
        "sk_genre_id",
        row_number().over(Window.orderBy("genre_name"))
    )
    .select("sk_genre_id", "genre_name")
)

# Grava a dimensão
(
    dim_genres.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_genres")
)

print(
    "Dimensão de gêneros:",
    spark.table("workspace.gold.dim_genres").count(),
    "registros"
)

# Relaciona filmes e gêneros
bridge_movie_genres = (
    df_filmes_generos.alias("fg")
    .join(
        df_filmes.alias("f"),
        trim(col("fg.id")) == trim(col("f.movie_id")),
        "inner"
    )
    .join(
        dim_genres.alias("g"),
        lower(trim(col("fg.genero"))) ==
        lower(trim(col("g.genre_name"))),
        "inner"
    )
    .select(
        col("f.sk_movie_id"),
        col("g.sk_genre_id")
    )
    .dropDuplicates()
)

# Grava a ligação
(
    bridge_movie_genres.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.bridge_movie_genres")
)

print(
    "Ligação entre filmes e gêneros:",
    spark.table("workspace.gold.bridge_movie_genres").count(),
    "registros"
)

display(spark.table("workspace.gold.bridge_movie_genres").limit(10))


In [0]:
# Cria a ligação entre filmes e pessoas

from pyspark.sql.functions import col, trim, lower

# Carrega as tabelas
df_filmes = spark.table("workspace.gold.dim_movies")
df_pessoas = spark.table("workspace.gold.dim_people")
df_creditos = spark.table("workspace.silver.tb_pessoas_empresas")

# Relaciona filmes, pessoas e funções
bridge_movie_people = (
    df_creditos.alias("c")
    .filter(col("c.tipo").isin("ator", "diretor", "roteirista"))
    .join(
        df_filmes.alias("f"),
        trim(col("c.id")) == trim(col("f.movie_id")),
        "inner"
    )
    .join(
        df_pessoas.alias("p"),
        trim(col("c.nome")) == trim(col("p.person_name")),
        "inner"
    )
    .select(
        col("f.sk_movie_id"),
        col("p.sk_person_id"),
        col("c.tipo").alias("role")
    )
    .dropDuplicates()
)

# Grava a ligação
(
    bridge_movie_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.bridge_movie_people")
)

print(
    "Ligação entre filmes e pessoas:",
    spark.table("workspace.gold.bridge_movie_people").count(),
    "registros"
)

display(spark.table("workspace.gold.bridge_movie_people").limit(10))


In [0]:
# Cria a ligação entre filmes e empresas

from pyspark.sql.functions import col, trim

# Carrega as tabelas

df_filmes = spark.table("workspace.gold.dim_movies")
df_empresas = spark.table("workspace.gold.dim_companies")
df_creditos = spark.table("workspace.silver.tb_pessoas_empresas")

# Relaciona filmes e empresas produtoras

bridge_movie_companies = (
    df_creditos.alias("c")
    .filter(col("c.tipo") == "empresa")
    .join(
        df_filmes.alias("f"),
        trim(col("c.id")) == trim(col("f.movie_id")),
        "inner"
    )
    .join(
        df_empresas.alias("e"),
        trim(col("c.nome")) == trim(col("e.company_name")),
        "inner"
    )
    .select(
        col("f.sk_movie_id"),
        col("e.sk_company_id")
    )
    .dropDuplicates()
)

# Grava a ligação

(
    bridge_movie_companies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.bridge_movie_companies")
)

print(
    "Ligação entre filmes e empresas:",
    spark.table("workspace.gold.bridge_movie_companies").count(),
    "registros"
)

display(spark.table("workspace.gold.bridge_movie_companies").limit(10))



In [0]:
# Cria a tabela fato de filmes

from pyspark.sql import Window
from pyspark.sql.functions import col, row_number, desc_nulls_last

# Carrega as tabelas

df_filmes = spark.table("workspace.gold.dim_movies")
df_financeiro = spark.table("workspace.silver.tb_financeiro_filmes")
df_metricas = spark.table("workspace.silver.tb_metricas_engajamento")

# Seleciona o registro financeiro mais recente de cada filme

janela_financeiro = Window.partitionBy("id").orderBy(
    desc_nulls_last("ingestion_datetime"),
    desc_nulls_last("revenue"),
    desc_nulls_last("budget")
)

df_financeiro_unico = (
    df_financeiro
    .withColumn("rn", row_number().over(janela_financeiro))
    .filter(col("rn") == 1)
    .select("id", "budget", "revenue")
)

# Seleciona as métricas mais recentes de cada filme

janela_metricas = Window.partitionBy("id").orderBy(
    desc_nulls_last("ingestion_datetime"),
    desc_nulls_last("popularity"),
    desc_nulls_last("vote_average"),
    desc_nulls_last("vote_count"),
    desc_nulls_last("averageRating"),
    desc_nulls_last("numVotes")
)

df_metricas_unicas = (
    df_metricas
    .withColumn("rn", row_number().over(janela_metricas))
    .filter(col("rn") == 1)
    .select(
        "id",
        "popularity",
        "vote_average",
        "vote_count",
        "averageRating",
        "numVotes"
    )
)

# Relaciona os filmes às informações financeiras e métricas

fact_movies = (
    df_filmes.alias("f")
    .join(
        df_financeiro_unico.alias("fin"),
        col("f.movie_id") == col("fin.id"),
        "left"
    )
    .join(
        df_metricas_unicas.alias("m"),
        col("f.movie_id") == col("m.id"),
        "left"
    )
    .select(
        col("f.sk_movie_id"),
        col("fin.budget").alias("budget_usd"),
        col("fin.revenue").alias("revenue_usd"),
        (col("fin.revenue") - col("fin.budget")).alias("profit_usd"),
        col("m.popularity"),
        col("m.vote_average"),
        col("m.vote_count"),
        col("m.averageRating"),
        col("m.numVotes")
    )
)

# Atualiza a tabela fato na Gold

(
    fact_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fact_movies")
)

print(
    "Tabela fato atualizada:",
    spark.table("workspace.gold.fact_movies").count(),
    "registros"
)

display(spark.table("workspace.gold.fact_movies").limit(10))


In [0]:
# Calcula o faturamento em BRL pela cotação mais recente disponível

from pyspark.sql.functions import (
    col, sum as spark_sum, round as spark_round, desc
)

# Obtém a cotação de referência
cotacao = (
    spark.table("workspace.silver.tb_cotacao_dolar")
    .filter(col("cotacao_venda").isNotNull())
    .orderBy(
        col("data_cotacao").desc(),
        col("data_hora_cotacao").desc()
    )
    .select("data_cotacao", "cotacao_venda")
    .first()
)

if cotacao is None:
    raise ValueError("Não há cotação de venda disponível na Silver.")

data_referencia = cotacao["data_cotacao"]
valor_cotacao = cotacao["cotacao_venda"]

print("Data da cotação de referência:", data_referencia)
print("Cotação de venda (BRL por USD):", valor_cotacao)

# Calcula o faturamento total

resultado_faturamento = (
    spark.table("workspace.gold.fact_movies")
    .filter(col("revenue_usd").isNotNull())
    .agg(
        spark_sum("revenue_usd").alias("faturamento_total_usd")
    )
    .select(
        spark_round(
            col("faturamento_total_usd"), 2
        ).alias("faturamento_total_usd"),
        spark_round(
            col("faturamento_total_usd") * valor_cotacao, 2
        ).alias("faturamento_total_brl")
    )
)

display(resultado_faturamento)




In [0]:
# Exibe os 5 filmes com maior popularidade

from pyspark.sql.functions import col

top_5_popularidade = (
    spark.table("workspace.gold.fact_movies").alias("f")
    .join(
        spark.table("workspace.gold.dim_movies").alias("m"),
        col("f.sk_movie_id") == col("m.sk_movie_id"),
        "inner"
    )
    .filter(col("f.popularity").isNotNull())
    .select(
        col("m.movie_id"),
        col("m.title"),
        col("m.release_date"),
        col("f.popularity")
    )
    .orderBy(col("popularity").desc(), col("movie_id").asc())
    .limit(5)
)

display(top_5_popularidade)


In [0]:
# Conta a quantidade de filmes por gênero

from pyspark.sql.functions import col, countDistinct

filmes_por_genero = (
    spark.table("workspace.gold.bridge_movie_genres").alias("b")
    .join(
        spark.table("workspace.gold.dim_genres").alias("g"),
        col("b.sk_genre_id") == col("g.sk_genre_id"),
        "inner"
    )
    .groupBy(col("g.genre_name"))
    .agg(
        countDistinct("b.sk_movie_id").alias("quantidade_filmes")
    )
    .orderBy(
        col("quantidade_filmes").desc(),
        col("genre_name").asc()
    )
)

display(filmes_por_genero)



In [0]:
# Exibe os 10 filmes com maior faturamento

from pyspark.sql.functions import col

top_10_faturamento = (
    spark.table("workspace.gold.fact_movies").alias("f")
    .join(
        spark.table("workspace.gold.dim_movies").alias("m"),
        col("f.sk_movie_id") == col("m.sk_movie_id"),
        "inner"
    )
    .filter(col("f.revenue_usd").isNotNull())
    .select(
        col("m.movie_id"),
        col("m.title"),
        col("m.release_date"),
        col("f.revenue_usd")
    )
    .orderBy(
        col("revenue_usd").desc(),
        col("movie_id").asc()
    )
    .limit(10)
)

display(top_10_faturamento)


In [0]:
# Identifica o ator com mais filmes nos últimos dois anos

from pyspark.sql.functions import (
    col, countDistinct, current_date, add_months,
    max as spark_max, lit
)

# Usa como referência a data de lançamento mais recente da base,
# desconsiderando lançamentos futuros e filmes não lançados.

df_filmes = spark.table("workspace.gold.dim_movies")

data_referencia = (
    df_filmes
    .filter(
        col("release_date").isNotNull() &
        (col("release_date") <= current_date()) &
        (col("status").isin("Released", "Lançado"))
    )
    .agg(spark_max("release_date").alias("data_maxima"))
    .first()["data_maxima"]
)

if data_referencia is None:
    print("Não há filmes lançados com data válida para esta análise.")
else:
    # Conta os filmes de cada ator nos 24 meses anteriores
    # à data de referência encontrada na própria base.

    participacoes_atores = (
        spark.table("workspace.gold.bridge_movie_people").alias("b")
        .filter(col("b.role") == "ator")
        .join(
            df_filmes.alias("m"),
            col("b.sk_movie_id") == col("m.sk_movie_id"),
            "inner"
        )
        .join(
            spark.table("workspace.gold.dim_people").alias("p"),
            col("b.sk_person_id") == col("p.sk_person_id"),
            "inner"
        )
        .filter(
            (col("m.release_date") >= add_months(lit(data_referencia), -24)) &
            (col("m.release_date") <= lit(data_referencia)) &
            (col("m.status").isin("Released", "Lançado"))
        )
        .groupBy(
            col("p.sk_person_id"),
            col("p.person_name")
        )
        .agg(
            countDistinct("b.sk_movie_id").alias("quantidade_filmes")
        )
        .orderBy(
            col("quantidade_filmes").desc(),
            col("p.person_name").asc()
        )
        .limit(10)
    )

    print("Data de referência:", data_referencia)
    display(participacoes_atores)
    

In [0]:
# Identifica as empresas com maior lucro nos últimos cinco anos

from pyspark.sql.functions import (
    col, countDistinct, sum as spark_sum,
    current_date, add_months, round as spark_round,
    max as spark_max, lit
)

# Encontra a data de lançamento mais recente da base,
# desconsiderando lançamentos futuros e filmes não lançados.

df_filmes = spark.table("workspace.gold.dim_movies")

data_referencia = (
    df_filmes
    .filter(
        col("release_date").isNotNull() &
        (col("release_date") <= current_date()) &
        (col("status").isin("Released", "Lançado"))
    )
    .agg(spark_max("release_date").alias("data_maxima"))
    .first()["data_maxima"]
)

if data_referencia is None:
    print("Não há filmes lançados com data válida para esta análise.")
else:
    # Soma o lucro dos filmes lançados nos 60 meses anteriores
    # à data de referência encontrada na própria base.

    lucro_empresas = (
        spark.table("workspace.gold.bridge_movie_companies").alias("b")
        .join(
            spark.table("workspace.gold.dim_companies").alias("c"),
            col("b.sk_company_id") == col("c.sk_company_id"),
            "inner"
        )
        .join(
            df_filmes.alias("m"),
            col("b.sk_movie_id") == col("m.sk_movie_id"),
            "inner"
        )
        .join(
            spark.table("workspace.gold.fact_movies").alias("f"),
            col("b.sk_movie_id") == col("f.sk_movie_id"),
            "inner"
        )
        .filter(
            (col("m.release_date") >= add_months(lit(data_referencia), -60)) &
            (col("m.release_date") <= lit(data_referencia)) &
            (col("m.status").isin("Released", "Lançado")) &
            col("f.revenue_usd").isNotNull() &
            col("f.budget_usd").isNotNull() &
            (col("f.revenue_usd") > 0) &
            (col("f.budget_usd") > 0)
        )
        .groupBy(
            col("c.sk_company_id"),
            col("c.company_name")
        )
        .agg(
            countDistinct("b.sk_movie_id").alias("quantidade_filmes"),
            spark_round(
                spark_sum(col("f.profit_usd")), 2
            ).alias("lucro_total_usd")
        )
        .orderBy(
            col("lucro_total_usd").desc(),
            col("company_name").asc()
        )
        .limit(10)
    )

    print("Data de referência:", data_referencia)
    display(lucro_empresas)
    

In [0]:
# Cria a tabela de contexto dos filmes para GenAI

from pyspark.sql.functions import (
    col, collect_set, concat_ws, concat, lit, coalesce,
    when, trim, year, format_number
)

# Carrega as tabelas Gold

df_filmes = spark.table("workspace.gold.dim_movies")
df_fato = spark.table("workspace.gold.fact_movies")
df_ponte_pessoas = spark.table("workspace.gold.bridge_movie_people")
df_pessoas = spark.table("workspace.gold.dim_people")

# Agrupa os atores de cada filme

atores_por_filme = (
    df_ponte_pessoas.alias("b")
    .filter(col("b.role") == "ator")
    .join(
        df_pessoas.alias("p"),
        col("b.sk_person_id") == col("p.sk_person_id"),
        "inner"
    )
    .groupBy(col("b.sk_movie_id"))
    .agg(
        concat_ws(", ", collect_set(col("p.person_name"))).alias("atores")
    )
)

# Agrupa os diretores de cada filme

diretores_por_filme = (
    df_ponte_pessoas.alias("b")
    .filter(col("b.role") == "diretor")
    .join(
        df_pessoas.alias("p"),
        col("b.sk_person_id") == col("p.sk_person_id"),
        "inner"
    )
    .groupBy(col("b.sk_movie_id"))
    .agg(
        concat_ws(", ", collect_set(col("p.person_name"))).alias("diretores")
    )
)

# Cria um texto por filme, tratando os campos ausentes

gold_genai_movies_context = (
    df_filmes.alias("m")
    .join(
        df_fato.alias("f"),
        col("m.sk_movie_id") == col("f.sk_movie_id"),
        "left"
    )
    .join(
        atores_por_filme.alias("a"),
        col("m.sk_movie_id") == col("a.sk_movie_id"),
        "left"
    )
    .join(
        diretores_por_filme.alias("d"),
        col("m.sk_movie_id") == col("d.sk_movie_id"),
        "left"
    )
    .select(
        col("m.movie_id").alias("movie_id"),
        col("m.title").alias("title"),
        concat(
            lit("O filme "),
            coalesce(col("m.title"), lit("de título não informado")),
            lit(", lançado no ano de "),
            coalesce(
                year(col("m.release_date")).cast("string"),
                lit("ano não informado")
            ),
            lit(", faturou "),
            coalesce(
                concat(lit("US$ "), format_number(col("f.revenue_usd"), 2)),
                lit("um valor não informado")
            ),
            lit(" e teve um custo de "),
            coalesce(
                concat(lit("US$ "), format_number(col("f.budget_usd"), 2)),
                lit("valor não informado")
            ),
            lit(". Estrelado por "),
            when(
                col("a.atores").isNotNull() & (trim(col("a.atores")) != ""),
                col("a.atores")
            ).otherwise(lit("atores não informados")),
            lit(" e dirigido por "),
            when(
                col("d.diretores").isNotNull() & (trim(col("d.diretores")) != ""),
                col("d.diretores")
            ).otherwise(lit("diretor não informado")),
            lit(", o filme possui a seguinte sinopse: "),
            when(
                col("m.overview").isNotNull() & (trim(col("m.overview")) != ""),
                col("m.overview")
            ).otherwise(lit("Sinopse não informada")),
            lit(".")
        ).alias("llm_context_document")
    )
)

# Grava a tabela de contexto na Gold

(
    gold_genai_movies_context.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_genai_movies_context")
)

print(
    "Tabela de contexto GenAI gravada:",
    spark.table("workspace.gold.gold_genai_movies_context").count(),
    "registros"
)

display(
    spark.table("workspace.gold.gold_genai_movies_context").limit(10)
)

